# Practical 3: Pixel-Level Spatial GeoAI and GPS Drone Scouting
### Hands-on Session: Trouble Spots, Strong Spots, Fertilizer Advice and Farmer-in-the-Loop Intelligence with Google Gemini

In this practical:
- Ingest spatial telemetry derived from the 30-meter pixel raster cube.
- Find both the **trouble spots** and the **strongest, healthiest spots** in every plot, with Plot IDs and GPS coordinates.
- Add **farmer-provided field information** that satellites cannot see (fertilizer history, variety, soil, budget).
- Generate a precision Field Scouting, Fertilizer and Ground Action Plan.
- Let the AI **ask the farmer for missing information** instead of guessing, then give refined advice once the farmer answers.

## Step 1: Import required libraries

In [1]:
import os
import sys
import json
import pandas as pd
from IPython.display import display, Markdown
from dotenv import load_dotenv
from google import genai

## Step 2: Initialize Google GenAI Client with your API key from .env

In [2]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
model_name = os.getenv("GEOAI_MODEL", "gemini-3.5-flash-lite")

client = genai.Client(api_key=gemini_api_key)
print("Google GenAI Client successfully initialized.")
print("Target Model:", model_name)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Google GenAI Client successfully initialized.
Target Model: gemini-3.5-flash-lite


## Step 3: Ingest pixel-level spatial anomaly telemetry
Loading pre-computed spatial anomalies for all plots: Plot IDs, problem sectors, and exact GPS coordinates of the worst spot.

In [ ]:
df_scout = pd.read_parquet("data/rice_farm_scout_anomalies.parquet")
df_scout['healthy_pixel_percent'] = (100 - df_scout['bad_pixel_percent']).round(1)

print("Pixel-Level Spatial Anomaly Telemetry:")
display(df_scout)

## Step 4: Find the strongest, healthiest spot in every plot
Precision farming is not only about fixing bad patches. We also locate where the crop is doing best, so the farmer can learn from those areas.

Using the peak-growth layer of the pixel Zarr cube and the plot boundaries, we find for each plot:
- The GPS location of its strongest pixel.
- How vigorous its canopy was at peak growth compared with the rest of the farm (NDVI).
- How much nitrogen its leaves had at peak growth compared with the rest of the farm (NDRE), a useful signal for fertilizer planning.

We also read the time series to work out the crop's current growth stage, because fertilizer advice depends on it.

In [ ]:
import xarray as xr
import geopandas as gpd

pixel_ds = xr.open_zarr("data/rice_farm_pixel_cube.zarr")
plots = gpd.read_file("farm_boundary/farm_boundary.geojson")[['plot_id', 'geometry']]

# Pixels at peak growth, tagged with the plot they fall in
peak = pixel_ds.sel(time='Peak_Vegetative')[['NDVI', 'NDRE']].to_dataframe().reset_index().dropna(subset=['NDVI', 'NDRE'])
pts = gpd.GeoDataFrame(peak, geometry=gpd.points_from_xy(peak['lon'], peak['lat']), crs="EPSG:4326")
pts = gpd.sjoin(pts, plots, predicate='within')

best = pts.loc[pts.groupby('plot_id')['NDVI'].idxmax(), ['plot_id', 'lat', 'lon']]
best = best.rename(columns={'lat': 'best_spot_latitude', 'lon': 'best_spot_longitude'}).round(5)
plot_stats = pts.groupby('plot_id')[['NDVI', 'NDRE']].median().reset_index()

# Plain-language labels relative to the rest of the farm, so the AI does not quote decimals
plot_stats['crop_vigour_at_peak'] = pd.qcut(plot_stats['NDVI'], 3, labels=['weaker than farm average', 'about farm average', 'among the strongest on the farm'])
plot_stats['leaf_nitrogen_at_peak'] = pd.qcut(plot_stats['NDRE'], 3, labels=['low', 'moderate', 'good'])

plot_data = (df_scout.merge(best, on='plot_id').merge(plot_stats.drop(columns=['NDVI', 'NDRE']), on='plot_id')
             .rename(columns={'bad_pixel_percent': 'trouble_area_percent', 'healthy_pixel_percent': 'healthy_area_percent'})
             [['plot_id', 'healthy_area_percent', 'trouble_area_percent', 'crop_vigour_at_peak', 'leaf_nitrogen_at_peak',
               'problem_sector', 'worst_spot_latitude', 'worst_spot_longitude', 'best_spot_latitude', 'best_spot_longitude']]
             .sort_values('healthy_area_percent', ascending=False))

# Current crop stage from the farm-wide time series
df_ts = pd.read_parquet("data/rice_farm_timeseries.parquet")
farm_ndvi = df_ts.groupby('date')['NDVI'].mean()
past_peak = farm_ndvi.iloc[-1] < 0.8 * farm_ndvi.max()
crop_stage = {
    "peak_growth_date": farm_ndvi.idxmax().strftime('%Y-%m-%d'),
    "latest_observation_date": farm_ndvi.index[-1].strftime('%Y-%m-%d'),
    "current_stage": "past peak growth, crop is ripening and drying down" if past_peak else "still in active growth",
}

scout_data = plot_data.astype({'crop_vigour_at_peak': str, 'leaf_nitrogen_at_peak': str}).to_dict(orient='records')

print("Crop stage:", json.dumps(crop_stage, indent=2))
display(plot_data)

## Step 5: Add farmer-provided field information
Satellites see the crop canopy, water and vegetation stress. They cannot see what fertilizer was applied, which variety was planted, the soil type, or the farmer's budget. Correct fertilizer and management advice needs this ground truth.

Fill in what you know. Leave anything you do not know as `None`. The AI is instructed to **ask the farmer** for missing details it needs, rather than guess.

In [ ]:
farm_inputs = {
    "crop": "Rice",
    "variety": None,                     # e.g. "FARO 44"
    "planting_date": None,               # e.g. "2024-06-20"
    "fertilizer_already_applied": None,  # e.g. "NPK 15-15-15 at 2 bags/ha at planting, Urea 1 bag/ha at tillering"
    "soil_type": None,                   # e.g. "clay loam"
    "irrigation": None,                  # e.g. "canal-fed, can drain plots individually"
    "budget_for_inputs": None,           # e.g. "about 150,000 naira for this season"
    "farmer_observations": None,         # e.g. "yellow leaves in Plot_05, rats near the southern edge"
}


def known_inputs():
    return {k: v for k, v in farm_inputs.items() if v is not None}


def missing_inputs():
    return [k for k, v in farm_inputs.items() if v is None]


print("Farmer has provided:", json.dumps(known_inputs(), indent=2))
print("Still unknown:", missing_inputs())

## Step 6: Generate Live Field Scouting, Fertilizer and Ground Action Plan
Google Gemini translates pixel telemetry and farmer information into a practical plan covering both weak and strong areas, with exact Plot IDs and GPS waypoints.

In [ ]:
ADVISOR_RULES = """Do not use technical math jargon or decimal index numbers. Talk about practical farm reality (stunted seedlings, water pools, weed patches, soil leveling, leaf colour).
Only use the satellite data and the information the farmer has provided. Never invent facts about the farm (such as fertilizer already applied, variety or soil type).
If correct advice depends on information listed as unknown, say clearly what you can already tell from the satellite data, then ask the farmer up to 3 short, specific questions under a heading "Questions for the Farmer", explaining in one line why each answer matters.
Do not use emojis."""

prompt = f"""You are an expert Chief Farm Agronomist briefing a {farm_inputs['crop'].lower()} farmer and drone scout on their field today.
{ADVISOR_RULES}

Crop stage from satellite time series:
{json.dumps(crop_stage, indent=2)}

Plot-by-plot satellite telemetry (healthy and trouble areas, crop vigour and leaf nitrogen at peak growth, GPS of worst and best spots):
{json.dumps(scout_data, indent=2)}

Information provided by the farmer:
{json.dumps(known_inputs(), indent=2)}
Unknown so far: {missing_inputs()}

Write a direct Field Scouting and Action Plan:
1. Priority Plots with Trouble Spots: Name the top 3 plots with trouble spots (Plot ID and which corner or sector), what the scout is likely to see on the ground, and the GPS waypoint.
2. Best Performing Plots: Name the top 3 healthiest plots and give the GPS of their strongest spot. Explain what is likely going right there and what the farmer should check on the ground so the same practice can be copied to the weak plots.
3. Fertilizer Plan: Based on the crop stage and leaf nitrogen per plot, say plainly whether fertilizer is worth applying now. If yes, which plots, what type and roughly how much; if no, say why and what to plan for next season (which plots need more, which need less).
4. Immediate Ground Remedy: Simple instructions for workers at the trouble spots (e.g. dig a small channel to drain trapped water, spot-spray weeds).
5. Questions for the Farmer: Only if information you need is missing.

Format in clean Markdown with clear headings and bold Plot IDs."""

print("Calling Google Gemini GeoAI API for spatial scout report...")
response = client.models.generate_content(model=model_name, contents=prompt)
display(Markdown(response.text))

## Step 7: Define interactive Q&A helper for field queries
The helper always reads the latest `farm_inputs`, so when the farmer answers the AI's questions, the next answer improves.

In [ ]:
def ask_scout_ai(question):
    p = f"""You are an experienced {farm_inputs['crop']} Farm Operations Expert talking to a practical commercial farmer.
Always mention the specific Plot IDs, physical sectors (like southern corner, eastern side), and GPS coordinates where appropriate.
{ADVISOR_RULES}

Crop stage: {json.dumps(crop_stage)}
Field telemetry: {json.dumps(scout_data)}
Information provided by the farmer: {json.dumps(known_inputs())}
Unknown so far: {missing_inputs()}

Farmer asks: "{question}"
Answer directly and practically in 2-3 short paragraphs, followed by "Questions for the Farmer" only if needed."""
    r = client.models.generate_content(model=model_name, contents=p)
    return r.text

print("ask_scout_ai function ready.")

## Step 8: Trouble Spot Question 1: Finding the Worst Bad Patch
*Which specific plot has the worst patch of sick or stunted crops, and which corner should my farm workers walk to?*

In [6]:
q1 = "Which specific plot has the worst patch of sick or stunted crops, and which corner should my farm workers walk to?"
print("Farmer asks:", q1)
display(Markdown(ask_scout_ai(q1)))

Farmer asks: Which specific plot has the worst patch of sick or stunted crops, and which corner should my farm workers walk to?


Plot 05 has the worst overall patch of stunted crops on the entire farm, showing significant stress across fourteen percent of its canopy. Send your crew straight to the southern-eastern corner of that plot. Have them walk directly to GPS coordinates latitude 7.88929, longitude 8.3386 to inspect the root zone and check for localized drainage failure or nutrient deficiency.

Your second priority is Plot 01, where about eleven percent of the crop is struggling in the southern-eastern corner as well. Direct your workers to inspect the worst spot there at latitude 7.88312, longitude 8.32974. Look closely at these specific zones for stem borer damage or uneven fertilizer spread during top-dressing.

## Step 9: Trouble Spot Question 2: Exact Phone GPS Waypoints
*Give me the exact phone GPS waypoints I can give to my tractor driver or field scout to find the weak spots immediately.*

In [7]:
q2 = "Give me the exact phone GPS waypoints I can give to my tractor driver or field scout to find the weak spots immediately."
print("Farmer asks:", q2)
display(Markdown(ask_scout_ai(q2)))

Farmer asks: Give me the exact phone GPS waypoints I can give to my tractor driver or field scout to find the weak spots immediately.


Here are the exact coordinates for your field scout and tractor driver to locate the weak spots across your struggling plots. Send these straight to their phones.

The highest priority is **Plot_05**, which is showing the worst stress at latitude 7.88929, longitude 8.3386 in the southern-eastern corner. Right nearby in **Plot_01**, tell them to check latitude 7.88312, longitude 8.32974, also in the southern-eastern sector. For **Plot_02**, the trouble spot is just a short distance away at latitude 7.8842, longitude 8.33135 in that same southern-eastern zone. 

Move next to **Plot_15**, where the crew needs to inspect the southern-western corner at latitude 7.88446, longitude 8.33215. For **Plot_06**, send them back to the southern-eastern corner at latitude 7.89117, longitude 8.32195. Finally, check the northern-western corner of **Plot_12** at latitude 7.90136, longitude 8.32436. Have them check these exact drops for water pooling, nutrient deficiency, or early pest pressure.

## Step 10: Good Area Question: Learning from the Best Plots
*Which plots are doing best and where exactly is the strongest crop? What should I check there so I can copy it to my weak plots?*

In [ ]:
q3 = "Which plots are doing best and where exactly is the strongest crop? What should I check there so I can copy it to my weak plots?"
print("Farmer asks:", q3)
display(Markdown(ask_scout_ai(q3)))

## Step 11: Fertilizer Question: Where, What and How Much
*Which plots need fertilizer, what type and how much should I apply, and where exactly? Which plots can I skip to save money?*

With most farm details still unknown, expect the AI to give what it can from the satellite data and then ask the farmer for the missing details.

In [ ]:
q4 = "Which plots need fertilizer, what type and how much should I apply, and where exactly? Which plots can I skip to save money?"
print("Farmer asks:", q4)
display(Markdown(ask_scout_ai(q4)))

## Step 12: Farmer Answers the AI and Gets Refined Advice
Fill in the answers to the AI's questions below (example answers are shown; replace them with your real farm details), then ask the fertilizer question again. The advice now combines satellite intelligence with ground truth from the farmer.

In [ ]:
farm_inputs.update({
    "variety": "FARO 44",
    "planting_date": "2024-06-20",
    "fertilizer_already_applied": "NPK 15-15-15 at 2 bags/ha at planting, no top dressing yet",
    "soil_type": "clay loam",
    "budget_for_inputs": "about 150,000 naira for this season",
    "farmer_observations": "yellowing leaves in the south-eastern part of Plot_05",
})

print("Still unknown:", missing_inputs())
print("Farmer asks again:", q4)
display(Markdown(ask_scout_ai(q4)))